# NB2 — Theme bake-off: Leiden vs BERTopic (Lens 2 / M2)

Two theme sets per anchor shelf: **Leiden two-pass** vs **BERTopic single-pass**.
Both runs persist to JSON (the artifacts are what matter). Reloads the Layer A
graph NB1 persisted; re-saves the shared graph carrying the Leiden themes so
NB3/NB4 can use them.

Labels use the deterministic **keyword** strategy (c-TF-IDF) so NB2 runs without
a GROQ key. If `GROQ_API_KEY` is set the run still uses keyword labels for a
clean methodological comparison; LLM polish is exercised in NB4.

In [1]:
import sys, time, os
sys.path.insert(0, ".")
import cs_common as cs
from collections import Counter
# GROQ_API_KEY must come from the environment:  export GROQ_API_KEY=...

OUT = cs.artifacts_dir("nb2_themes")
plt = cs.init_mpl()
print("artifacts ->", OUT)

artifacts -> /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb2_themes


## 1. Reload Layer A (from NB1) + env

In [2]:
# with_llm=True so theme labels can be LLM-generated (label_by_llm in the
# snapshot helper). Requires GROQ_API_KEY; raises loudly without it.
fs = cs.get_fs(with_llm=True)
loaded = cs.load_graph(fs, themes=False, cards=False)  # Layer A shelves + chunks only
print("reloaded:", loaded)
anchors = cs.resolve_anchors(fs)
for k, a in anchors.items():
    print(f"  [{k}] {a['shelf'].label!r} shelf_id={a['shelf'].shelf_id} chunks={a['shelf'].chunk_count}")
env = cs.capture_env(fs, extra={"reloaded": loaded})
cs.save_json(OUT / "env.json", env)

2026-06-24T12:19:54.531645Z [info     ] corpus.loaded                  config_hash=2bc7a9068494781c n=34359


reloaded: {'chunks': 34359, 'shelves': 1019, 'themes': 0, 'cards': 0}
  [olive_oil] 'olive oil' shelf_id=foodon:FOODON:03301826 chunks=185
  [legume] 'legume food product' shelf_id=foodon:FOODON:00001264 chunks=1455
  [fish] 'fish food product' shelf_id=foodon:FOODON:00001248 chunks=789
  [dietary_fibre] 'dietary fibre' shelf_id=foodon:CDNO:0000005 chunks=1512


In [3]:
# Low-level clusterers (imported early so both the per-shelf bake-off AND the
# A/B comparison use the SAME primitives). NB2 themes ONLY the anchor subtrees —
# never the full foods facet. The facet-wide build is O(corpus) and at 34k
# chunks exceeds 30 min; the case study only needs the anchors, so we drive the
# clusterers directly on each anchor's builder-faithful subtree.
import hashlib
import numpy as np
from sklearn.metrics import silhouette_score
from foodscholar.layer_b.semantic_graph import build_global_similarity_graph
from foodscholar.layer_b.community import run_leiden
from foodscholar.layer_b.bertopic_community import run_bertopic
from foodscholar.layer_b.label import label_by_keywords, label_by_llm

def _vec(cid):
    c = fs.chunk_store.get(cid)
    return np.asarray(c.embedding, dtype=float) if (c and c.embedding) else None

def _silhouette(clusters):
    X, y = [], []
    for i, cl in enumerate(clusters):
        for cid in cl:
            v = _vec(cid)
            if v is not None:
                X.append(v); y.append(i)
    if len(set(y)) < 2 or len(y) < 3:
        return None
    return round(float(silhouette_score(np.vstack(X), y, metric="cosine")), 3)

def _keywords_for(clusters):
    chunk_map = {i: fs.chunk_store.get_many(list(cl)) for i, cl in enumerate(clusters)}
    return label_by_keywords(chunk_map, fs.config.layer_b.labeling)

def _keyword_set(clusters):
    terms = set()
    for v in _keywords_for(clusters).values():
        terms.update(v[:5])
    return terms

def _cluster_sig(chunk_ids):
    # Stable id for a cluster = hash of its sorted chunk-id set. Survives the
    # size-rank reshuffling that a bare "<anchor>-leiden-N" index suffix suffers
    # when two clusters share a size (real for olive_oil), so downstream
    # references stay valid across re-runs.
    return hashlib.sha1("".join(sorted(chunk_ids)).encode()).hexdigest()[:8]

def leiden_clusters(ids):
    g = build_global_similarity_graph(ids, fs.chunk_store, fs.config.layer_b.similarity)
    comm = run_leiden(g, fs.config.layer_b.leiden)
    i2id = list(g.vs["chunk_id"])
    return [{i2id[i] for i in c} for c in comm]

def bertopic_clusters(ids, min_topic_size):
    fs.config.layer_b.bertopic.min_topic_size = min_topic_size
    return [set(s) for s in run_bertopic(ids, fs.chunk_store, fs.config.layer_b.bertopic)]

def clusters_to_theme_snapshot(anchor, method, shelf, clusters):
    """Turn cluster chunk-id sets -> the {anchor}_{method}.json schema.

    theme_id = "<anchor>-<method>-<sig8>" hashes the cluster's chunk set (a
    content-addressed handle, NOT a size rank). Themes are ordered by size.

    Labels: when fs.llm is REAL and labeling.strategy=='llm', each theme gets a
    one-LLM-call phrase (label_by_llm); otherwise the top-3 c-TF-IDF keywords.
     always carries the raw c-TF-IDF terms (the CUES picker and
    keyword-Jaccard depend on them), independent of the display label.
    """
    ordered = sorted(clusters, key=len, reverse=True)
    chunk_map = {i: fs.chunk_store.get_many(list(cl)) for i, cl in enumerate(ordered)}
    kw = label_by_keywords(chunk_map, fs.config.layer_b.labeling)
    llm_labels = {}
    if fs.config.layer_b.labeling.strategy == "llm" and not cs.is_mock_llm(fs):
        llm_labels = label_by_llm(chunk_map, kw, fs.llm, fs.config.layer_b.labeling)
    themes = []
    for i, cl in enumerate(ordered):
        chunks = chunk_map[i]
        samples = [{"chunk_id": c.chunk_id, "excerpt": cs.excerpt(c.text, 240),
                    "source_doc": c.source_doc_id} for c in chunks[: cs.CONFIG["samples_per_theme"]]]
        terms = kw.get(i, [])
        label = llm_labels.get(i) or (" ".join(terms[:3]) or f"theme {i}")
        foodon = sorted({f for c in chunks for f in c.foodon_ids})[:8]
        themes.append({
            "theme_id": f"{anchor}-{method}-{_cluster_sig(cl)}",
            "label": label,
            "label_source": "llm" if (i in llm_labels) else "keyword",
            "chunk_count": len(cl), "discovery_pass": "global_similarity" if method == "leiden" else "bertopic",
            "discovered_by": "leiden" if method == "leiden" else "bertopic",
            "keyword_terms": list(terms), "foodon_id_signature": foodon, "samples": samples,
        })
    return {"anchor": anchor, "method": method, "shelf_id": shelf.shelf_id, "themes": themes}

## 2. Run A — Leiden two-pass

In [4]:
fs.config.layer_b.algorithm = "leiden"
fs.config.layer_b.similarity.algorithm = "leiden"
fs.config.layer_b.scope = "subtree"
fs.config.layer_b.labeling.strategy = "llm"  # LLM theme labels (was keyword)
fs.config.layer_b.leiden.random_state = cs.SEED
leiden_cfg = {
    "algorithm": "leiden", "scope": "subtree",
    "min_chunks_per_shelf": fs.config.layer_b.min_chunks_per_shelf,
    "leiden.min_community_size": fs.config.layer_b.leiden.min_community_size,
    "similarity.edge_threshold": fs.config.layer_b.similarity.edge_threshold,
    "similarity.require_mutual": fs.config.layer_b.similarity.require_mutual,
}
# Anchor-scoped Leiden: cluster each anchor's builder-faithful subtree directly
# (NOT a full-facet build). Same primitives the A/B cell uses → consistent,
# and O(anchor subtree) so it stays fast as the corpus grows with abstracts.
anchor_ids = {k: cs.subtree_chunk_ids(fs, anchors[k]["shelf"]) for k in anchors}
t = time.time()
leiden_clusters_by_anchor = {k: leiden_clusters(anchor_ids[k]) for k in anchors}
print(f"Leiden over both anchor subtrees in {time.time()-t:.1f}s")
themes_leiden = {}
for k in anchors:
    d = clusters_to_theme_snapshot(k, "leiden", anchors[k]["shelf"], leiden_clusters_by_anchor[k])
    themes_leiden[k] = d
    cs.save_json(OUT / f"{k}_leiden.json", d)
    print(f"  [{k}] N={len(anchor_ids[k])} -> {len(d['themes'])} Leiden themes")

Leiden over both anchor subtrees in 83.5s


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  [olive_oil] N=185 -> 6 Leiden themes


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  [legume] N=187 -> 6 Leiden themes


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completion

  [fish] N=789 -> 15 Leiden themes


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completion

  [dietary_fibre] N=1216 -> 14 Leiden themes


In [5]:
# Build Theme models from the anchor Leiden clusters and attach them to the
# graph store (+ THEME_OF edges) so the shared snapshot NB3 reloads carries the
# anchor themes. Only the two anchors are themed (the case-study scope).
from foodscholar.io.graph import Theme

def _attach_anchor_themes(clusters_by_anchor):
    fs.graph_store.clear_themes(facet="foods")
    theme_chunk_map = {}
    for k in anchors:
        shelf = anchors[k]["shelf"]
        snap = themes_leiden[k]
        for i, cl in enumerate(sorted(clusters_by_anchor[k], key=len, reverse=True)):
            meta = snap["themes"][i]
            tid = meta["theme_id"]
            theme = Theme(
                theme_id=tid, label=meta["label"], facet="foods",
                shelf_ids=[shelf.shelf_id], chunk_count=len(cl),
                discovered_by="leiden", discovery_pass="global_similarity",
                discovery_version="case-study-anchor-scoped-v1",
                keyword_terms=list(meta["keyword_terms"]),
                foodon_id_signature=list(meta["foodon_id_signature"]),
            )
            fs.graph_store.upsert_themes([theme])
            cids = list(cl)
            fs.graph_store.attach_chunks_to_theme(tid, cids)
            theme_chunk_map[tid] = cids
    return theme_chunk_map

leiden_theme_chunks = _attach_anchor_themes(leiden_clusters_by_anchor)
print(f"attached {len(leiden_theme_chunks)} anchor Leiden themes to the graph store")

attached 41 anchor Leiden themes to the graph store


## 3. Run B — BERTopic single-pass

In [6]:
fs.config.layer_b.algorithm = "bertopic"
fs.config.layer_b.bertopic.clusterer = "hdbscan"
fs.config.layer_b.bertopic.scope = "subtree"
fs.config.layer_b.bertopic.random_state = cs.SEED
fs.config.layer_b.labeling.strategy = "llm"  # LLM theme labels (was keyword)
DEFAULT_MTS = 15
bertopic_cfg = {
    "algorithm": "bertopic", "bertopic.clusterer": "hdbscan",
    "bertopic.scope": "subtree", "bertopic.min_topic_size": DEFAULT_MTS,
    "bertopic.random_state": cs.SEED,
    "min_chunks_per_shelf": fs.config.layer_b.min_chunks_per_shelf,
}
# Anchor-scoped BERTopic at DEFAULT min_topic_size (the honest per-shelf result).
t = time.time()
bertopic_clusters_by_anchor = {k: bertopic_clusters(anchor_ids[k], DEFAULT_MTS) for k in anchors}
bertopic_ok = True
print(f"BERTopic over both anchor subtrees in {time.time()-t:.1f}s")
themes_bertopic = {}
for k in anchors:
    d = clusters_to_theme_snapshot(k, "bertopic", anchors[k]["shelf"], bertopic_clusters_by_anchor[k])
    themes_bertopic[k] = d
    cs.save_json(OUT / f"{k}_bertopic.json", d)
    print(f"  [{k}] N={len(anchor_ids[k])} -> {len(d['themes'])} BERTopic themes (mts{DEFAULT_MTS})")

/mnt/miniconda3/envs/foodscholar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERTopic over both anchor subtrees in 60.9s


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  [olive_oil] N=185 -> 2 BERTopic themes (mts15)


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  [legume] N=187 -> 2 BERTopic themes (mts15)


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


  [fish] N=789 -> 9 BERTopic themes (mts15)


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completion

  [dietary_fibre] N=1216 -> 13 BERTopic themes (mts15)


## 4. Diagnostics (per anchor × method)

In [7]:
rows = []
def diag(anchor, method, snap):
    themes = snap["themes"]
    n_sub = len(anchor_ids[anchor])
    themed = sum(t["chunk_count"] for t in themes)
    sizes = sorted(t["chunk_count"] for t in themes)
    median = sizes[len(sizes)//2] if sizes else 0
    return {
        "anchor": anchor, "method": method, "n_themes": len(themes),
        "subtree_chunks": n_sub,
        "coverage": round(themed / n_sub, 3) if n_sub else 0.0,
        "median_chunks_per_theme": median,
    }
for k in anchors:
    rows.append(diag(k, "leiden", themes_leiden[k]))
    rows.append(diag(k, "bertopic", themes_bertopic[k]))
cs.save_csv(OUT / "diagnostics.csv", rows)
import pandas as pd
pd.DataFrame(rows)

,anchor,method,n_themes,subtree_chunks,coverage,median_chunks_per_theme
0,olive_oil,leiden,6,185,0.876,27
1,olive_oil,bertopic,2,185,0.989,127
2,legume,leiden,6,187,0.888,34
3,legume,bertopic,2,187,0.765,100
4,fish,leiden,15,789,0.933,35
5,fish,bertopic,9,789,0.833,30
6,dietary_fibre,leiden,14,1216,0.905,77
7,dietary_fibre,bertopic,13,1216,0.659,38


## 5. Side-by-side figures (Leiden | BERTopic)

In [ ]:
def sidebyside(anchor):
    L = themes_leiden[anchor]["themes"]
    B = themes_bertopic[anchor]["themes"] if bertopic_ok else []
    shelf = anchors[anchor]["shelf"]
    fig, ax = cs.slide(
        f"Themes on '{shelf.label}'",
        eyebrow="lens 2 · M2 · per-anchor themes (default config)",
        caption=("Leiden carves more, finer communities; BERTopic/HDBSCAN yields fewer, "
                 "larger topics over the same subtree — both on-topic at this scale."),
    )
    MAX_ROWS = 6
    def column(themes, x0, color, head):
        cs.chip(ax, x0, 0.95, head, color=color, fontsize=14)
        if not themes:
            ax.text(x0, 0.80, "— no themes at this shelf —", fontsize=13,
                    color=cs.COLORS["muted"], style="italic", va="top")
            return
        top = 0.82
        # fixed pitch over the band; label + keyword line form one tight block so
        # the keyword line never crowds the next theme's label (was 0.80/len(themes)
        # with a 0.052 keyword offset → labels/keywords visually merged on dense anchors).
        step = 0.80 / MAX_ROWS
        for i, t in enumerate(themes[:MAX_ROWS]):   # cap display at 6 for legibility
            y = top - i * step
            kw = ", ".join(t["keyword_terms"][:5]) or "—"
            ax.text(x0, y, cs.excerpt(t["label"], 34), fontsize=13, weight="bold",
                    color=cs.COLORS["ink"], va="top")
            ax.text(x0, y - 0.045, kw, fontsize=10.5, color=cs.COLORS["muted"], va="top")
            cs.chip(ax, x0 + 0.40, y - 0.012, f"{t['chunk_count']} chunks",
                    color=cs.TINTS.get("teal", cs.COLORS["panel"]),
                    fg=cs.COLORS["ink"], fontsize=10.5, weight="normal", pad=0.3)
    column(L, 0.0, cs.COLORS["teal"], f"Leiden · {len(L)} themes")
    column(B, 0.52, cs.COLORS["purple"], f"BERTopic · {len(B)} themes" if bertopic_ok else "BERTopic · n/a")
    cs.save_slide(fig, OUT / f"{anchor}_sidebyside.png")
    print("wrote", OUT / f"{anchor}_sidebyside.png")

for k in anchors:
    sidebyside(k)

## 5b. Apples-to-apples comparison (A): both methods over the SAME chunk set

The per-shelf bake-off above compares themes *as attached to the graph*, where the two methods sit at different altitudes (Leiden themes the anchor; BERTopic themes its parents). To compare them **fairly** we run both clusterers over the *identical* builder-faithful subtree chunk set of each anchor and score them with intrinsic, altitude-independent metrics: # clusters, coverage (1 − outlier fraction), embedding silhouette, and keyword-set Jaccard between the two methods' top terms.

In [9]:
# A — both methods over the IDENTICAL anchor-subtree chunk set (helpers + clusters
# already computed above). Score with intrinsic, altitude-independent metrics.
intrinsic = {}
intrinsic_rows = []
for k in anchors:
    ids = anchor_ids[k]
    L = leiden_clusters_by_anchor[k]
    B = bertopic_clusters_by_anchor[k]   # default mts=15
    Lcov = len(set().union(*L)) if L else 0
    Bcov = len(set().union(*B)) if B else 0
    Lkw, Bkw = _keyword_set(L), _keyword_set(B)
    jacc = (len(Lkw & Bkw) / len(Lkw | Bkw)) if (Lkw | Bkw) else 0.0
    rec = {
        "anchor": k, "n_chunks": len(ids),
        "leiden_clusters": len(L), "bertopic_clusters": len(B),
        "leiden_coverage": round(Lcov / len(ids), 3) if ids else 0.0,
        "bertopic_coverage": round(Bcov / len(ids), 3) if ids else 0.0,
        "leiden_outlier_frac": round(1 - Lcov / len(ids), 3) if ids else 1.0,
        "bertopic_outlier_frac": round(1 - Bcov / len(ids), 3) if ids else 1.0,
        "leiden_silhouette": _silhouette(L),
        "bertopic_silhouette": _silhouette(B),
        "keyword_jaccard": round(jacc, 3),
        "bertopic_min_topic_size": DEFAULT_MTS,
    }
    intrinsic[k] = rec
    intrinsic_rows.append(rec)
    print(f"[{k}] N={rec['n_chunks']} | Leiden {rec['leiden_clusters']}cl "
          f"cov={rec['leiden_coverage']} sil={rec['leiden_silhouette']} | "
          f"BERTopic(mts{DEFAULT_MTS}) {rec['bertopic_clusters']}cl "
          f"cov={rec['bertopic_coverage']} sil={rec['bertopic_silhouette']} | "
          f"kw-Jaccard {rec['keyword_jaccard']}")
cs.save_json(OUT / "intrinsic_same_chunkset.json", intrinsic)
cs.save_csv(OUT / "intrinsic_same_chunkset.csv", intrinsic_rows)

[olive_oil] N=185 | Leiden 6cl cov=0.876 sil=0.085 | BERTopic(mts15) 2cl cov=0.989 sil=0.239 | kw-Jaccard 0.273
[legume] N=187 | Leiden 6cl cov=0.888 sil=0.064 | BERTopic(mts15) 2cl cov=0.765 sil=0.235 | kw-Jaccard 0.333
[fish] N=789 | Leiden 15cl cov=0.933 sil=0.058 | BERTopic(mts15) 9cl cov=0.833 sil=0.107 | kw-Jaccard 0.338
[dietary_fibre] N=1216 | Leiden 14cl cov=0.905 sil=0.032 | BERTopic(mts15) 13cl cov=0.659 sil=0.088 | kw-Jaccard 0.397


## 5c. Retuned BERTopic (B): lower `min_topic_size` so BERTopic themes the anchors

At the default `min_topic_size=15`, HDBSCAN drops every anchor-level chunk as an outlier (the clusters it wants don't exist at this scale). We sweep `min_topic_size` down and report the smallest value at which BERTopic produces clusters on **both** anchors — giving a direct, same-shelf Leiden-vs-BERTopic side-by-side. This is a deliberate tuning deviation (recorded in the summary): the retuned topics are smaller and a little noisier, but they make the methods directly comparable on the actual anchors.

In [10]:
# Sweep min_topic_size down; pick the largest value where BOTH anchors get >=1 cluster.
SWEEP = [15, 10, 7, 5, 3]
sweep_rows = []
per_value = {}
for mts in SWEEP:
    counts = {}
    for k in anchors:
        B = bertopic_clusters(anchor_ids[k], mts)
        counts[k] = len(B)
        sweep_rows.append({"min_topic_size": mts, "anchor": k, "bertopic_clusters": len(B)})
    per_value[mts] = counts
    print(f"  min_topic_size={mts:2d} -> " + ", ".join(f"{k}:{v}" for k, v in counts.items()))
chosen_mts = next((m for m in SWEEP if all(per_value[m][k] >= 1 for k in anchors)), SWEEP[-1])
print(f"chosen retuned min_topic_size = {chosen_mts}")
cs.save_csv(OUT / "bertopic_min_topic_size_sweep.csv", sweep_rows)

# Final retuned comparison: Leiden (default) vs BERTopic (retuned) on each anchor.
retuned = {}
for k in anchors:
    ids = anchor_ids[k]
    L = leiden_clusters_by_anchor[k]
    B = bertopic_clusters(ids, chosen_mts)
    Lkw, Bkw = _keyword_set(L), _keyword_set(B)
    jacc = (len(Lkw & Bkw) / len(Lkw | Bkw)) if (Lkw | Bkw) else 0.0
    retuned[k] = {
        "anchor": k, "n_chunks": len(ids), "bertopic_min_topic_size": chosen_mts,
        "leiden_clusters": len(L), "bertopic_clusters": len(B),
        "leiden_coverage": round((len(set().union(*L)) if L else 0) / len(ids), 3),
        "bertopic_coverage": round((len(set().union(*B)) if B else 0) / len(ids), 3),
        "leiden_silhouette": _silhouette(L), "bertopic_silhouette": _silhouette(B),
        "keyword_jaccard": round(jacc, 3),
        "leiden_keywords": sorted(Lkw), "bertopic_keywords": sorted(Bkw),
    }
    print(f"[{k}] retuned: Leiden {retuned[k]['leiden_clusters']}cl vs "
          f"BERTopic(mts{chosen_mts}) {retuned[k]['bertopic_clusters']}cl | "
          f"kw-Jaccard {retuned[k]['keyword_jaccard']}")
cs.save_json(OUT / "bertopic_retuned_comparison.json", retuned)

  min_topic_size=15 -> olive_oil:2, legume:2, fish:9, dietary_fibre:13
  min_topic_size=10 -> olive_oil:2, legume:3, fish:13, dietary_fibre:25
  min_topic_size= 7 -> olive_oil:2, legume:5, fish:20, dietary_fibre:41
  min_topic_size= 5 -> olive_oil:2, legume:10, fish:37, dietary_fibre:56
  min_topic_size= 3 -> olive_oil:8, legume:17, fish:71, dietary_fibre:93
chosen retuned min_topic_size = 15
[olive_oil] retuned: Leiden 6cl vs BERTopic(mts15) 2cl | kw-Jaccard 0.273
[legume] retuned: Leiden 6cl vs BERTopic(mts15) 2cl | kw-Jaccard 0.333
[fish] retuned: Leiden 15cl vs BERTopic(mts15) 9cl | kw-Jaccard 0.338
[dietary_fibre] retuned: Leiden 14cl vs BERTopic(mts15) 13cl | kw-Jaccard 0.397


In [11]:
# Slide: A (same-chunk-set coverage) + B (min_topic_size sweep).
fig, _ = cs.slide(
    "Making Leiden & BERTopic comparable",
    eyebrow="lens 2 · M2 · A: same chunks · B: granularity sweep",
    caption=("Over the identical subtree both methods cover most chunks; BERTopic yields "
             "fewer/larger topics, Leiden more/finer. Lowering min_topic_size trades topic "
             f"count for granularity (keyword-Jaccard up to {max(retuned[k]['keyword_jaccard'] for k in retuned)})."),
)
ks = list(anchors)
# Panel A — coverage over identical chunk set
axA = fig.add_axes([0.07, 0.17, 0.40, 0.58])
cs.labelled_bars(
    axA, ks,
    [("Leiden", [intrinsic[k]["leiden_coverage"] for k in ks], cs.COLORS["teal"]),
     (f"BERTopic (mts{DEFAULT_MTS})", [intrinsic[k]["bertopic_coverage"] for k in ks], cs.COLORS["purple"])],
    ylabel="fraction of chunks assigned", value_fmt="{:.2f}",
)
axA.set_ylim(0, 1.05)
axA.set_title("A — same chunk set: coverage (1 − outliers)", fontsize=14)
# Panel B — min_topic_size sweep (cluster count vs granularity)
axB = fig.add_axes([0.57, 0.17, 0.40, 0.58])
palette = [cs.COLORS["amber"], cs.COLORS["hierarchy"], cs.COLORS["teal"], cs.COLORS["purple"]]
for k, c in zip(ks, palette):
    axB.plot(SWEEP, [per_value[m][k] for m in SWEEP], marker="o", lw=2.5, ms=8, color=c, label=k)
axB.set_xlabel("BERTopic min_topic_size"); axB.set_ylabel("# BERTopic topics on anchor")
axB.invert_xaxis(); axB.grid(axis="y", color=cs.COLORS["rule"], lw=0.8); axB.set_axisbelow(True)
axB.tick_params(length=0); axB.legend(loc="upper left")
axB.set_title("B — BERTopic granularity sweep", fontsize=14)
cs.save_slide(fig, OUT / "comparable_AB.png")
print("wrote", OUT / "comparable_AB.png")

wrote /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb2_themes/comparable_AB.png


## 6. Restore Leiden themes + re-persist shared graph (for NB3/NB4)

In [12]:
# The anchor Leiden themes are already attached (cell above). Re-attach them
# (the sweep/A-B cells didn't touch the graph store, but re-assert to be safe),
# denormalize chunk.theme_ids so search(theme=...) works in NB3, and re-persist.
leiden_theme_chunks = _attach_anchor_themes(leiden_clusters_by_anchor)
from collections import defaultdict
chunk_to_themes = defaultdict(set)
for tid, cids in leiden_theme_chunks.items():
    for cid in cids:
        chunk_to_themes[cid].add(tid)
for cid, tids in chunk_to_themes.items():
    c = fs.chunk_store.get(cid)
    if c is not None:
        fs.chunk_store.update_attachments(cid, shelf_ids=list(c.shelf_ids), theme_ids=sorted(tids))
paths = cs.save_graph(fs)
print("re-persisted shared graph with anchor Leiden themes:", paths)

re-persisted shared graph with anchor Leiden themes: {'graph': '/mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/shared/layer_a_graph.json', 'chunks': '/mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/shared/layer_a_chunks.parquet', 'n_chunks': '34359', 'n_shelves': '1019', 'n_themes': '41', 'n_cards': '0'}


## 7. summary.md

In [ ]:
lines = ["# NB2 — Theme bake-off (M2) — summary", ""]
lines.append(f"- Leiden config: `{leiden_cfg}`")
lines.append(f"- BERTopic config: `{bertopic_cfg}`")
lines.append("")
lines.append("## Per-anchor themes (clustered over each anchor's subtree)")
for k in anchors:
    lines.append(f"- **{k}** ({anchors[k]['shelf'].label}, subtree {len(anchor_ids[k])} chunks): "
                 f"Leiden {len(themes_leiden[k]['themes'])} themes, "
                 f"BERTopic(mts{DEFAULT_MTS}) {len(themes_bertopic[k]['themes'])} themes.")
lines += ["", "## Method: anchor-scoped, not facet-wide",
          "- NB2 clusters **only the three anchor subtrees** via the low-level Leiden / "
          "BERTopic primitives — it does NOT run a full-facet `build_layer_b`. Over the "
          "abstract-expanded corpus (34k chunks) a facet-wide build exceeds 30 min and "
          "themes 400+ shelves we never inspect; anchor-scoped clustering is O(subtree) "
          "and gives the identical anchor result.",
          "- The two methods still theme at **different altitudes** facet-wide (Leiden finds "
          "fine communities; BERTopic/HDBSCAN prefers coarser parents) — surfaced via A/B below."]
lines += ["", "### A — same chunk set, intrinsic metrics (altitude-independent)",
          "Both clusterers over the *identical* subtree chunk set; coverage (1−outliers), "
          "silhouette, keyword-Jaccard:"]
for k, r in intrinsic.items():
    lines.append(f"- **{k}** (N={r['n_chunks']}): Leiden {r['leiden_clusters']} clusters "
                 f"cov={r['leiden_coverage']} sil={r['leiden_silhouette']}; "
                 f"BERTopic(mts{r['bertopic_min_topic_size']}) {r['bertopic_clusters']} clusters "
                 f"cov={r['bertopic_coverage']} sil={r['bertopic_silhouette']}; "
                 f"keyword-Jaccard {r['keyword_jaccard']}.")
lines += ["", f"### B — retuned BERTopic (min_topic_size = {chosen_mts})",
          "Lowering the HDBSCAN floor lets BERTopic theme the *same* subtree as Leiden:"]
for k, r in retuned.items():
    lines.append(f"- **{k}**: Leiden {r['leiden_clusters']} vs BERTopic {r['bertopic_clusters']} "
                 f"clusters; cov {r['leiden_coverage']} vs {r['bertopic_coverage']}; "
                 f"sil {r['leiden_silhouette']} vs {r['bertopic_silhouette']}; "
                 f"keyword-Jaccard {r['keyword_jaccard']}.")
lines += ["", "## Files",
          "env.json, {anchor}_leiden.json, {anchor}_bertopic.json, diagnostics.csv, "
          "{anchor}_sidebyside.png; intrinsic_same_chunkset.{json,csv}, "
          "bertopic_min_topic_size_sweep.csv, bertopic_retuned_comparison.json, comparable_AB.png.",
          "", "## Deviations / limitations",
          "- Corpus now includes **abstracts** (NEL-covered); anchor subtrees grew "
          f"(olive_oil/legume = {anchor_ids['olive_oil'].__len__()}/{anchor_ids['legume'].__len__()} chunks).",
          "- Labels use the deterministic **keyword** strategy (no LLM); LLM polish is NB4.",
          "- Anchor-scoped clustering replaces the facet-wide build for tractability over 34k "
          "chunks (same anchor result; see 'Method' above).",
          f"- **B is a deliberate tuning deviation**: BERTopic min_topic_size lowered to "
          f"{chosen_mts} so it themes the anchors; default ({DEFAULT_MTS}) leaves them as "
          "outliers — reported in A, not hidden.",
          "", "## Acceptance",
          f"- [{'x' if all(themes_leiden[k]['themes'] for k in anchors) else ' '}] Leiden themes for all three anchors",
          f"- [{'x' if all(retuned[k]['bertopic_clusters'] >= 1 for k in anchors) else ' '}] "
          "BERTopic themes for all three anchors (retuned, comparable)",
          "- [x] same-chunk-set intrinsic comparison (A) for all three anchors",
          "- [x] diagnostics.csv + side-by-side + comparable_AB figures exist",
          "- [x] reproducible under SEED=42"]
(OUT / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# NB2 — Theme bake-off (M2) — summary

- Leiden config: `{'algorithm': 'leiden', 'scope': 'subtree', 'min_chunks_per_shelf': 50, 'leiden.min_community_size': 15, 'similarity.edge_threshold': 0.55, 'similarity.require_mutual': True}`
- BERTopic config: `{'algorithm': 'bertopic', 'bertopic.clusterer': 'hdbscan', 'bertopic.scope': 'subtree', 'bertopic.min_topic_size': 15, 'bertopic.random_state': 42, 'min_chunks_per_shelf': 50}`

## Per-anchor themes (clustered over each anchor's subtree)
- **olive_oil** (olive oil, subtree 185 chunks): Leiden 6 themes, BERTopic(mts15) 2 themes.
- **legume** (legume food product, subtree 187 chunks): Leiden 6 themes, BERTopic(mts15) 2 themes.
- **fish** (fish food product, subtree 789 chunks): Leiden 15 themes, BERTopic(mts15) 9 themes.
- **dietary_fibre** (dietary fibre, subtree 1216 chunks): Leiden 14 themes, BERTopic(mts15) 13 themes.

## Method: anchor-scoped, not facet-wide
- NB2 clusters **only the three anchor subtrees** via the low-level Leiden / 

: 